# Test of system of unlinear equation

In [33]:
import numpy as np
import scipy as sp
import re
from scipy.optimize import fsolve, minimize, root

In [ ]:
# Number of control volumes in a row
M = 3
# Number of longitudinal rows
N = 2

T_hs_BC = 400  # Inlet temperature hot side
T_fg_BC = 200  # Inlet temperature flue gas side

# Initial guess for temperatures
T_hs_guess = np.array([T_hs_BC - i*4 for i in range(M*N)])
T_fg_guess = np.array([T_fg_BC + i*5 for i in range(N)])

T_hs_BC = 400  # Inlet temperature hot side
T_fg_BC = 200  # Inlet temperature flue gas side

T_fg_guess = np.array([T_fg_BC + i*5 for i in range(N)])
T_hs = np.concatenate((np.array([T_hs_BC]), T_hs_guess))


class node: 
    def __init__(self, num, temperature,type):
        self.num = num
        self.temperature = temperature
        self.type = type  # 'hs' or 'fg'
        
    def __repr__(self):
        return f"Node {self.num} of type {self.type}"


class control_volume:
    def __init__(self,num):
        self.num = num
        
        # Calculation coefficients
        N_cv = N * M  # Number of control volumes 

        # Hard coded for simplicity
        self.UA = 100 / N_cv 
        self.m_dot_hs = 0.05 / N
        self.m_dot_fg = 0.05 / M 

        self.c_p_hs = 1005 # J/kgK
        self.c_p_fg = 500 # J/kgK

    def set_nodes(self, node_hs_in, node_hs_out, node_fg_in, node_fg_out):
        self.node_hs_in = node_hs_in
        self.node_hs_out = node_hs_out
        self.node_fg_in = node_fg_in
        self.node_fg_out = node_fg_out
        
    def is_boundary_hs(self,T_hs_BC):
        self.is_boundary_hs = True
        self.node_hs_in.temperature = T_hs_BC
        
    def is_boundary_fg(self,T_fg_BC):
        self.is_boundary_fg = True
        self.node_fg_in.temperature = T_fg_BC
    
    def calculate(self):
        # Energy balance for hot side
        Q_hs = self.m_dot_hs * self.c_p_hs * (self.node_hs_in.temperature - self.node_hs_out.temperature)
        # Energy balance for flue gas side
        Q_fg = self.m_dot_fg * self.c_p_fg * (self.node_fg_out.temperature - self.node_fg_in.temperature)
        # Heat exchanged
        Q_out = self.UA * ((self.node_hs_in.temperature + self.node_hs_out.temperature)/2 - (self.node_fg_in.temperature + self.node_fg_out.temperature)/2)
        return Q_hs, Q_fg, Q_out    
    
    def __repr__(self):
        return f"Control Volume {self.num}"
    
class system:
    def __init__(self, M, N):
        self.M = M
        self.N = N
        
        # Create control volumes
        # Control volumes are numbered column wise 
        # CV[M][N] where M is number of control volumes in a row and N is number of rows
            # So for a 2x2 system control volume 0 is at (0,0), control volume 1 is at (1,0), control volume 2 is at (0,1) etc.  
        self.control_volumes = np.array([[control_volume(i*M+j) for j in range(M)] for i in range(N)])
        
        # Create nodes and assign to control volumes
        self._create_nodes()
    
    def _create_nodes(self):
        '''
        Create nodes for the system and assign them to control volumes. Each output node is the input node for the next control volume.
        For the flue gas side, the same node is shared between control volumes in the same row.
        '''
        
        hs_node_num = 0
        fg_node_num = 0
        
        for row_control_volume in self.control_volumes:
            if row_control_volume[0].num == 0:
                # First control volume in the system create both inlet and outlet nodes
                node_fg_in = node(fg_node_num, None, 'fg')
                fg_node_num += 1
                node_fg_out = node(fg_node_num, None, 'fg')
                fg_node_num += 1
            
            for control_volume in row_control_volume:
                if control_volume.num == 0:
                    # First control volume in the system create both inlet and outlet nodes
                    node_hs_in = node(hs_node_num, None, 'hs')
                    hs_node_num += 1
                    node_hs_out = node(hs_node_num, None, 'hs')
                    hs_node_num += 1
                
                # set nodes
                control_volume.set_nodes(node_hs_in, node_hs_out, node_fg_in, node_fg_out)
                
                # Prepare nodes for next control volume
                node_hs_in = node_hs_out
                node_hs_out = node(hs_node_num, None, 'hs')
                hs_node_num += 1
            
            # Prepare nodes for next row of control volumes
            node_fg_in = node_fg_out
            node_fg_out = node(fg_node_num, None, 'fg')
            fg_node_num += 1
            
    
    def set_boundary_conditions(self, T_hs_BC, T_fg_BC):
        '''
        Set boundary conditions for the system. T_hs_BC is the inlet temperature for the hot side. T_fg_BC is the inlet temperature for the flue gas side.
        '''
        # Set boundary conditions for hot side
        self.control_volumes[0,0].is_boundary_hs(T_hs_BC)
        # Set boundary conditions for flue gas side
        for j in range(self.N):
            self.control_volumes[j,0].is_boundary_fg(T_fg_BC)
    
    def update_temp(self,T_guess):
        '''
        Updates the temperatures of the nodes in the system based on the provided temperature guesses.
        T_guess is a concatenated list of T_hs and T_fg guesses without boundary conditions.
        '''
        # Unpack temperature guesses
        
        # There is always the same number of hot side temperatures as control volumes
        # Since for the first row the inlet temperature is known from BCs
        # And for subsequent rows the outlet temperature of the previous row is the inlet temperature
        T_hs_guess = T_guess[:(self.M*self.N)]
        T_fg_guess = T_guess[(self.M*self.N):]
        
        # Update hot side temperatures
        for control_volume in self.control_volumes.flatten():
            if control_volume.is_boundary_hs:
                control_volume.node_hs_out.temperature = T_hs_guess[control_volume.num]
            else:
                control_volume.node_hs_in.temperature = T_hs_guess[control_volume.num]
                control_volume.node_hs_out.temperature = T_hs_guess[control_volume.num+1]
            
        # Update flue gas side temperatures
        for row_num,row in enumerate(self.control_volumes):
            print( row)
            for control_volume in row:
                if control_volume.is_boundary_fg:
                    control_volume.node_fg_out.temperature = T_fg_guess[row_num]
                else:
                    control_volume.node_fg_in.temperature = T_fg_guess[row_num]
                    control_volume.node_fg_out.temperature = T_fg_guess[row_num+1]

    def solve_system(self,T_hs_BC,T_fg_BC):
        '''
        Solve the system of equations to find the temperatures of the nodes.
        '''
        
        self.set_boundary_conditions(T_hs_BC, T_fg_BC)
        # Initial guess for temperatures
        T_hs_guess = np.array([T_hs_BC - i*4 for i in range(self.M*self.N)])
        T_fg_guess = np.array([T_fg_BC + i*5 for i in range(self.N)])
        
        T_guess = np.concatenate((T_hs_guess, T_fg_guess))
        
        def equations(T_guess):
            self.update_temp(T_guess)
            eqs = []
            for control_volume in self.control_volumes.flatten():
                Q_hs, Q_fg, Q_out = control_volume.calculate()
                eqs.append(Q_hs - Q_out)
                eqs.append(Q_fg - Q_out)
            return eqs
        
        sol = root(equations, T_guess, method='hybr')
        return sol
    
    # Functions to display the system
    def show_system(self):
        # Create a copy to manipulate for display
        control_volumes = self.control_volumes.copy()
        
        # Flip every second column to represent snaking flow path
        control_volumes[1::2, :] = np.flip(control_volumes[1::2, :], axis=1)
                
        print("System Control Volumes:")
        for row in control_volumes:
            print(*row,sep='\t')
            
    def show_nodes(self):
        '''
        Show all nodes in the system. in an array format with snaking flow path.
        '''
        # Create a copy to manipulate for display
        control_volumes = self.control_volumes.copy()
        # Flip every second column to represent snaking flow path
        control_volumes[1::2, :] = np.flip(control_volumes[1::2, :], axis=1)
        
        print("\nSystem Nodes:")
        for row_idx, row in enumerate(control_volumes):
            row_matrix = []
            swap_hs = (row_idx % 2 == 1)
            for control_volume in row:
                hs_left = control_volume.node_hs_in.num
                hs_right = control_volume.node_hs_out.num
                if swap_hs:
                    hs_left, hs_right = hs_right, hs_left
                bold_cv = f"\x1b[4m{control_volume.num}\x1b[0m"
                
                # Create 3x3 matrix of nodes for each control volume to represent the connections
                # Layout: fg_in (top), hs_in (left), CV (center), hs_out (right), fg_out (bottom)
                matrix = np.array([["", control_volume.node_fg_in.num, ""],
                                  [hs_left, bold_cv, hs_right],
                                  ["", control_volume.node_fg_out.num, ""]])
                
                # Concatenate matrices horizontally
                row_matrix.append(matrix)
            for i in range(len(row_matrix[0])):
                line_cells = [cell for block in row_matrix for cell in block[i]]
                string_cells = ["" if cell == "" else str(cell) for cell in line_cells]
                cell_lengths = [len(re.sub(r"\x1b\[[0-9;]*m", "", cell)) for cell in string_cells]
                max_width = max(cell_lengths, default=1)
                line = " ".join(cell.rjust(max_width) if cell else "".rjust(max_width) for cell in string_cells)
                print(line.rstrip())
                
    def show_temperatures(self):
        '''
        Show all the temperatures of the nodes in the system. in an array format with snaking flow path.
        '''
        # Create a copy to manipulate for display
        control_volumes = self.control_volumes.copy()
        # Flip every second column to represent snaking flow path
        control_volumes[1::2, :] = np.flip(control_volumes[1::2, :], axis=1)
        
        print("\nSystem Node Temperatures:")
        for row_idx, row in enumerate(control_volumes):
            row_matrix = []
            swap_hs = (row_idx % 2 == 1)
            for control_volume in row:
                hs_left_temp = control_volume.node_hs_in.temperature
                hs_right_temp = control_volume.node_hs_out.temperature
                if swap_hs:
                    hs_left_temp, hs_right_temp = hs_right_temp, hs_left_temp
                bold_cv = f"\x1b[4m{control_volume.num}\x1b[0m"
                
                # Create 3x3 matrix of temperatures for each control volume to represent the connections
                # Layout: fg_in (top), hs_in (left), CV (center), hs_out (right), fg_out (bottom)
                matrix = np.array([["", f"{control_volume.node_fg_in.temperature:.1f}", ""],
                                  [f"{hs_left_temp:.1f}", bold_cv, f"{hs_right_temp:.1f}"],
                                  ["", f"{control_volume.node_fg_out.temperature:.1f}", ""]])
                
                # Concatenate matrices horizontally
                row_matrix.append(matrix)
            for i in range(len(row_matrix[0])):
                line_cells = [cell for block in row_matrix for cell in block[i]]
                string_cells = ["" if cell == "" else str(cell) for cell in line_cells]
                cell_lengths = [len(re.sub(r"\x1b\[[0-9;]*m", "", cell)) for cell in string_cells]
                max_width = max(cell_lengths, default=1)
                line = " ".join(cell.rjust(max_width) if cell else "".rjust(max_width) for cell in string_cells)
                print(line.rstrip())


sys = system(M,N)
sys.show_system()
sys.show_nodes()
T_hs_guess = np.zeros(M*N)
T_fg_guess = np.ones(N)
T_guess = np.concatenate((T_hs_guess, T_fg_guess))
sys.set_boundary_conditions(10, 5)
sys.update_temp(T_guess)

sys.show_temperatures()

sys.solve_system(T_hs_BC,T_fg_BC)

System Control Volumes:
Control Volume 0	Control Volume 1	Control Volume 2
Control Volume 5	Control Volume 4	Control Volume 3

System Nodes:
  0     0     0
0 0 1 1 1 2 2 2 3
  1     1     1
  1     1     1
6 5 5 5 4 4 4 3 3
  2     2     2
[Control Volume 0 Control Volume 1 Control Volume 2]
[Control Volume 3 Control Volume 4 Control Volume 5]

System Node Temperatures:
    5.0         5.0         5.0
10.0 0  0.0  0.0 1  0.0  0.0 2  0.0
    1.0         1.0         1.0
    1.0         1.0         1.0
0.0 5 0.0 0.0 4 0.0 0.0 3 0.0
    1.0         1.0         1.0


TypeError: 'bool' object is not callable

In [ ]:
import numpy as np 
import scipy as sp
import matplotlib.pyplot as plt
from enum import StrEnum
from functools import cached_property 

import pyfluids as pf
from pyfluids import FluidsList, Input
# Set the units system to SI with Celsius
pf.PyFluidsConfig.units_system = pf.UnitsSystem.SIWithCelsius

g = 9.81 # m/s^2

# Enum classes for various types
class FlowType(StrEnum):
    '''Enum for flow types'''
    Internal = "internal"
    External = "external"
    
class arrangementType(StrEnum):
    '''Enum for tube arrangement types'''
    Inline      = "Inline"
    Staggered   = "Staggered"
    Single_Tube = "Single_Tube"
    Single_Row  = "Single_Row"
    
class finType(StrEnum):
    '''Enum for fin types'''
    Rectangular = "rectangular"
    Circular    = "circular"
    Spiral      = "spiral"

# Classes for heat exchanger modeling
class Geometry:
    '''Class representing the geometry of the heat exchanger with given dimensions'''
    class _Tube:
        '''Inner class representing tube geometry'''
        def __init__(self):
            self.geom_set = False
        def set_geometry(self,outer_diameter,wall_thickness):
            '''Sets the geometry of the tube
            args:   
                outer_diameter: Outer diameter of the tube (m)
                wall_thickness: Wall thickness of the tube (m)
            '''
            self.geom_set       = True
            self.outer_diameter = outer_diameter # 
            self.wall_thickness = wall_thickness # Tube wall thickness

            # Derived properties
            self.inner_diameter     = self.outer_diameter - 2 * self.wall_thickness
            self.inner_perimeter    = np.pi * self.inner_diameter
            self.outer_perimeter    = np.pi * self.outer_diameter
            self.inner_area         = np.pi * (self.inner_diameter/2)**2

    class _Duct:
        '''Inner class representing duct geometry'''
        def __init__(self):
            self.geom_set = False
        def set_geometry(self,a,b):
            '''Sets the geometry of the duct
            args:
                a: Duct width (m)
                b: Duct height (m)
            '''
            self.geom_set = True
            self.a  = a #height
            self.b  = b #width 

            # Derived properties
            self.area   = self.a * self.b # Frontal cross setional area of duct
    
    class _Fin:
        '''Inner class representing fin geometry'''
        def __init__(self):
            self.geom_set = False
        def set_geometry(self,average_fin_thickness,fin_height,fin_spacing,fin_type=finType.Rectangular):
            '''Sets the geometry of the fin
            args:
                average_fin_thickness: Average thickness of the fin (m)
                fin_height: Height of the fin measured from tube OD (m)
                fin_spacing: Spacing between fins / fin pitch (m)
                fin_type: Type of the fin from finType enum (default is finType.Rectangular)
            '''
            
            self.geom_set = True
            self.average_fin_thickness  = average_fin_thickness #delta_r
            self.fin_height             = fin_height #l_r
            self.fin_spacing            = fin_spacing #s_r
            self.fin_type               = fin_type

            # match fin_type:
            #     case finType.Rectangular:
            #         self.square_height = self.fin_height*2 + self.Tube.outer_diameter
            #     case finType.Circular:
            #         self.finning_diameter = self.Fin.fin_height*2 + self.Tube.outer_diameter
            #     case finType.Spiral:
            #         raise NotImplementedError("Spiral fin type not implemented yet.")

    class _Bank: 
        '''Inner class representing bank geometry'''
        def __init__(self):
            self.geom_set = False
        def set_geometry(self, transverse_number_of_rows,longitudinal_number_of_rows,transverse_pitch,longitudinal_pitch,L_ccrs,arrangement):
            ''''Sets the geometry of the tube bank
            args:
                transverse_number_of_rows: Number of tube rows in the bank transverse to flow direction (-)
                longitudinal_number_of_rows: Number of tube rows in the bank longitudinal to flow direction (-)
                transverse_pitch: Transverse pitch of tubes (m)
                longitudinal_pitch: Longitudinal pitch of tubes (m)
                arrangement: Arrangement type of tubes (Inline, Staggered, Single_Tube, Single_Row) (-)
            '''
            self.geom_set = True

            # Row numbers
            self.transverse_number_of_rows      = transverse_number_of_rows # z_1
            self.longitudinal_number_of_rows    = longitudinal_number_of_rows # z_2
            self.total_number_of_tubes          = self.transverse_number_of_rows * self.longitudinal_number_of_rows

            # Pitches
            self.transverse_pitch               = transverse_pitch # S1
            self.longitudinal_pitch             = longitudinal_pitch # S2
            self.diagonal_pitch                 = np.sqrt((1/4)*self.transverse_pitch**2 + self.longitudinal_pitch**2)

            # Arrangement
            self.arrangement                    = arrangement 
            self.L_ccrs                         = L_ccrs # Length of pipe at cross section

            # Derived properties
            # match arrangement:
            #     case arrangementType.Inline:
            #         self.diagonal_pitch = None
            #     case arrangementType.Staggered:
            #         self.diagonal_pitch = np.sqrt((1/4)*self.transverse_pitch**2 + self.longitudinal_pitch**2)
            #     case arrangementType.Single_Tube:
            #         self.diagonal_pitch = None
            #     case arrangementType.Single_Row:
            #         self.diagonal_pitch = None
    
    def __init__(self):
        # Initialize inner classes
        self.Tube = self._Tube()
        self.Duct = self._Duct()
        self.Fin  = self._Fin()
        self.Bank = self._Bank()

    @cached_property
    def finning_diameter(self):
        '''Calculates the finning diameter'''
        return self.Tube.outer_diameter+2*self.Fin.fin_height

    @cached_property
    def Conventional_diameter(self):
        # Conventional diameter of finned tube:
        d_cl    = self.Tube.outer_diameter + ((2*self.Fin.fin_height*self.Fin.average_fin_thickness)/(self.Fin.fin_spacing))
        return d_cl

    @cached_property
    def phi_parameter(self):
        # Conventional diameter of finned tube:
        d_cl = self.Conventional_diameter
        # phi parameter:
        phi_cl  = (self.Bank.transverse_pitch-d_cl)/(self.Bank.diagonal_pitch-d_cl)
        return phi_cl

    @cached_property
    def Free_flow_area(self):
        # Conventional diameter of finned tube:
        d_cl = self.Conventional_diameter
        # phi parameter
        phi_cl = self.phi_parameter
        # Free flow area:
        if phi_cl <= 2:
            F = self.Duct.a*self.Duct.b-self.Bank.transverse_number_of_rows*self.Bank.L_ccrs*d_cl
        elif phi_cl > 2:
            F = (self.Duct.a*self.Duct.b - self.Bank.transverse_number_of_rows*self.Bank.L_ccrs*d_cl)*(2/phi_cl)
        return F

    @cached_property
    def Equivalent_diameter(self):
        # phi parameter
        phi_cl  = self.phi_parameter
        
        # Equivalent diameter
        d_eq    = (2*(self.Fin.fin_spacing*(self.Bank.transverse_pitch-self.Tube.outer_diameter)-2*self.Fin.fin_height*self.Fin.average_fin_thickness))/(2*self.Fin.fin_height + self.Fin.fin_spacing)
        
        if phi_cl <= 2:
            d_eq = d_eq
        elif phi_cl > 2:
            d_eq = (2*d_eq)/phi_cl

        return d_eq

    @cached_property
    def A_total_over_F(self):
        # Atotal/F ratio:
        return (np.pi*(self.Tube.outer_diameter*self.Fin.fin_spacing+2*self.Fin.fin_height*self.Fin.average_fin_thickness+2*self.Fin.fin_height*(self.Fin.fin_height+self.Tube.outer_diameter)))/(self.Bank.transverse_pitch*self.Fin.fin_spacing-(self.Tube.outer_diameter*self.Fin.fin_spacing+2*self.Fin.fin_height*self.Fin.average_fin_thickness))

    @cached_property
    def S_1_over_S_2(self):
        # S_1 over S_2 ratio:
        return self.Bank.transverse_pitch/self.Bank.longitudinal_pitch

    @cached_property
    def sigma_1(self):
        '''Calculates the transverse pitch to diameter ratio'''
        return self.Bank.transverse_pitch / self.Tube.outer_diameter
    
    @cached_property
    def sigma_2(self):
        '''Calculates the longitudinal pitch to diameter ratio'''
        return self.Bank.longitudinal_pitch / self.Tube.outer_diameter
    
    @cached_property
    def Psi_r(self):
        '''Calculates the fin coefficient for finned tubes'''
        # In the below calculations we assume constant fin thickness so that the average fin thickness equals the fin thickness at base and tip
        match self.Fin.fin_type:
            case finType.Rectangular:
                Psi_r = (2*(self.Fin.square_height**2 - 0.785 * self.Tube.outer_diameter**2 + 2 * self.Fin.square_height*self.Fin.average_fin_thickness))(np.pi * self.Tube.outer_diameter * self.Fin.fin_spacing) + (1 - self.Fin.average_fin_thickness/self.Fin.fin_spacing)
            case finType.Circular: 
                Psi_r = 1/(2*self.Tube.outer_diameter*self.Fin.fin_spacing) *(self.Fin.finning_diameter**2 - self.Tube.outer_diameter**2 + 2*self.Fin.finning_diameter * self.Fin.average_fin_thickness) + (1- self.Fin.average_fin_thickness/self.Fin.fin_spacing)
    
    @cached_property
    def A_r(self):
        '''Calculates the heat transfer area for fins'''
        
        match self.Fin.fin_type:
            case finType.Rectangular:
                A_r = 2 * (self.Fin.square_height**2 - 0.785 * self.Tube.outer_diameter**2 + 2 * self.Fin.square_height*self.Fin.average_fin_thickness) * self.Bank.finned_tube_segment / self.Fin.fin_spacing * self.Bank.total_number_of_tubes
            case finType.Spiral | finType.Circular:
                A_r = np.pi/2 * (self.Fin.finning_diameter**2 - self.Tube.outer_diameter**2 + 2 * self.Fin.finning_diameter * self.Fin.average_fin_thickness) * self.Bank.finned_tube_segment / self.Fin.fin_spacing * self.Bank.total_number_of_tubes
        return A_r
        
    @cached_property
    def A_t(self):
        '''Calculates the heat transfer area for tubes'''
        raise NotImplementedError("Tube area calculation not implemented yet.")
    
    @cached_property
    def A(self):
        '''Calculates the total heat transfer area'''
        return self.A_r + self.A_t

class Node: # Fluid state at a specific position. 
    def __init__(self,fluid_type,x_pos,Input_1,Input_2):
        self.x_pos  = x_pos # m

        self.fluid  = pf.Fluid(fluid_type)
        self.fluid.update(Input_1, Input_2)

    def update_state(self, Input_1, Input_2):
        """Update thermodynamic state in-place."""
        self.fluid.update(Input_1, Input_2)

    def liquid_phase(self):
        """Return a Fluid object representing the saturated liquid phase."""
        if self.fluid.phase.name != "TwoPhase":
            raise ValueError("Node is not in a two-phase state")

        liquid = pf.Fluid(self.fluid.name)
        liquid.update(
            Input.pressure(self.fluid.pressure),
            Input.quality(0.0)
        )
        return liquid

    def vapor_phase(self):
        """Return a Fluid object representing the saturated vapor phase."""
        if self.fluid.phase.name != "TwoPhase":
            raise ValueError("Node is not in a two-phase state")

        vapor = pf.Fluid(self.fluid.name)
        vapor.update(
            Input.pressure(self.fluid.pressure),
            Input.quality(1.0)
        )
        return vapor

    def __repr__(self):
        return f"Fluid(type={self.fluid.name}, T={self.fluid.temperature} C, P={self.fluid.pressure} Pa, , H={self.fluid.enthalpy} J/kg, x={self.fluid.quality})"

class Internal: # Internal correlations
    def __init__(self,mass_flow,Node_in,Node_out,Geometry,Delta_x):
        self.Node_in    = Node_in
        self.Node_out   = Node_out
        self.Geometry   = Geometry
        self.m_dot      = mass_flow
        self.L          = Delta_x

    # ----- pressure drop ----- #

    def pressure_drop(self):
        '''Determines the type of pressure drop based on the flow type and phase change'''
        phase_in    = self.Node_in.fluid.phase.name
        phase_out   = self.Node_out.fluid.phase.name

        if phase_in in ("Liquid", "Gas") and phase_out in ("Liquid", "Gas"):
            dp = self.monophase_pressure_drop(self.Node_in.fluid,self.Node_out.fluid)

        elif phase_in == "TwoPhase" and phase_out == "TwoPhase":
            dp = self.two_phase_pressure_drop()

        elif phase_in != phase_out:
            # Phase change direction
            # Representative pressure drop for phase change segment
            if phase_in == "Gas" and phase_out == "TwoPhase":
                dp = self.monophase_pressure_drop(self.Node_in.fluid,self.Node_out.vapor_phase())
 
            elif phase_in == "TwoPhase" and phase_out == "Liquid":
                dp = self.monophase_pressure_drop(self.Node_in.liquid_phase(),self.Node_out.fluid)
                
        else:
            raise ValueError("Invalid phase change direction for pressure drop calculation")

        return dp
    
    def monophase_pressure_drop(self,Fluid_in,Fluid_out):
        print("Monophase check")

        # Calculate flow velocities
        u_in       = self.m_dot / (Fluid_in.density * self.Geometry.Tube.inner_area)
        u_out      = self.m_dot / (Fluid_out.density * self.Geometry.Tube.inner_area)

        # Calculate average properties
        rho_avg    = (Fluid_in.density + Fluid_out.density) / 2
        u_avg      = self.m_dot / (rho_avg * self.Geometry.Tube.inner_area)
        mu_avg     = (Fluid_in.dynamic_viscosity + Fluid_out.dynamic_viscosity) / 2           
        Re_avg     = self.Reynolds_number(rho_avg,u_avg,mu_avg,self.Geometry.Tube.inner_diameter)

        # Acceleration pressure drop
        DP_A        = 0.5 * (Fluid_out.density * u_out**2 - Fluid_in.density * u_in**2)
        
        # Frictional pressure drop (Darcy-Weisbach equation)
        # Calculate friction factor depending on flow regime
        zeta_fr = self.friction_factor(Re_avg)
        
        # Calculate frictional pressure drop
        DP_R    = zeta_fr * (self.L/self.Geometry.Tube.inner_diameter) * 0.5 * (rho_avg * u_avg**2)
        
        # Calculate gravitational pressure drop
        DP_G    = 0 # Assuming horizontal flow for now

        # Total pressure drop of internal segment
        DP_tot  = DP_A + DP_R + DP_G

        return(DP_tot)
    
    def two_phase_pressure_drop(self):
        print("two phase check")

        Node_in_liq     = self.Node_in.liquid_phase()
        Node_out_liq    = self.Node_out.liquid_phase()
        Node_in_vap     = self.Node_in.vapor_phase()
        Node_out_vap    = self.Node_out.vapor_phase()

        # Calculate average flow velocities
        u_liq_avg   = self.m_dot / (( (Node_in_liq.density + Node_out_liq.density) / 2) * self.Geometry.Tube.inner_area)
        u_vap_avg   = self.m_dot / (( (Node_in_vap.density + Node_out_vap.density) / 2) * self.Geometry.Tube.inner_area)

        # Calculate average properties
        rho_avg_liq = (Node_in_liq.density + Node_out_liq.density) / 2
        rho_avg_vap = (Node_in_vap.density + Node_out_vap.density) / 2
        mu_avg_liq  = (Node_in_liq.dynamic_viscosity + Node_out_liq.dynamic_viscosity) / 2
        mu_avg_vap  = (Node_in_vap.dynamic_viscosity + Node_out_vap.dynamic_viscosity) / 2

        # Calculate Reynolds numbers for liquid and vapor phases
        Re_avg_liq  = self.Reynolds_number(rho_avg_liq,u_liq_avg,mu_avg_liq,self.Geometry.Tube.inner_diameter)
        Re_avg_vap  = self.Reynolds_number(rho_avg_vap,u_vap_avg,mu_avg_vap,self.Geometry.Tube.inner_diameter)

        # Friction factor for liquid and vapor phases
        zeta_fr_liq = self.friction_factor(Re_avg_liq)
        zeta_fr_vap = self.friction_factor(Re_avg_vap)

        M_dot       = self.m_dot / self.Geometry.Tube.inner_area

        # Pressure drop of liquid and vapor phases
        DP_R_liq    = zeta_fr_liq * (M_dot**2)/(2 * rho_avg_liq * self.Geometry.Tube.inner_diameter)
        DP_R_vap    = zeta_fr_vap * (M_dot**2)/(2 * rho_avg_vap * self.Geometry.Tube.inner_diameter)

        # Assigning coefficients
        A = DP_R_liq
        B = DP_R_vap

        def F(A,B,x):
            F_ABx = -(3/4) * (1-x)**(3/4) * (A+2*(B-A)*x)+(1/4)*B*x**4 -9/14 * (B-A)*(1-x)**(7/3)
            return F_ABx
        
        DP_R = F(A,B,self.Node_out.fluid.quality) - F(A,B,self.Node_in.fluid.quality)

        return(DP_R)

    def friction_factor(self,Re):
        if 3000 < Re < 100000:
            zeta = 0.3614/((Re**(1/4)))
        elif 2*10**4 < Re < 2*10**6:
            zeta = 0.3614/((Re**(1/4)))
        elif Re > 10**6:
            f = lambda z: 1/z + 0.8 - 2*np.log(Re*np.sqrt(z))
            zeta = sp.optimize.fsolve(f, x0=0.001)[0]
        return zeta

    # ----- heat transfer ----- #

    def heat_transfer(self):

        phase_in    = self.Node_in.fluid.phase.name
        phase_out   = self.Node_out.fluid.phase.name        

        if phase_in in ("Liquid", "Gas") and phase_out in ("Liquid", "Gas"):
            h = self.monophase_heat_transfer(self.Node_in.fluid,self.Node_out.fluid)

        elif phase_in == "TwoPhase" and phase_out == "TwoPhase":
            h = self.twophase_heat_transfer(self.Node_in,self.Node_out)

        elif phase_in != phase_out:
            # Phase change direction
            # Representative pressure drop for phase change segment
            if phase_in == "Gas" and phase_out == "TwoPhase":
                print("check gas twophase")
                h = self.monophase_heat_transfer(self.Node_in.fluid,self.Node_out.vapor_phase())

            elif phase_in == "TwoPhase" and phase_out == "Liquid":
                print("check twophase liquid")
                h = self.monophase_heat_transfer(self.Node_in.liquid_phase(),self.Node_in.fluid)

            else:
                print("Error, boiling")

        return h
        
    def monophase_heat_transfer(self,Fluid_in,Fluid_out):
        
        # Average values:
        rho_avg     = (Fluid_in.density + Fluid_out.density) / 2
        u_avg       = self.m_dot / (rho_avg * self.Geometry.Tube.inner_area)
        mu_avg      = (Fluid_in.dynamic_viscosity + Fluid_out.dynamic_viscosity) / 2           
        cp_avg      = (Fluid_in.specific_heat + Fluid_out.specific_heat) / 2
        k_avg       = (Fluid_in.conductivity + Fluid_out.conductivity) / 2

        # Dimensionless number calculations:
        Re_avg      = self.Reynolds_number(rho_avg,u_avg,mu_avg,self.Geometry.Tube.inner_diameter)
        Pr_avg      = self.Prandtl_number(cp_avg,mu_avg,k_avg)

        n_tem       = 0.25
        C_tem       = (mu_avg/mu_avg)**n_tem # until wall temperature is decided

        # Colburn equation: 
        if 10000 > Re_avg:
            print("Error, Laminar flow")

        elif 10000 < Re_avg < 1*10**6 and 0.7 < Pr_avg < 2: 
            h       = 0.023 * k_avg/self.Geometry.Tube.inner_diameter * Re_avg**(0.8)*Pr_avg**(0.4) * C_tem
        
        elif 4*10**3 < Re_avg < 5*10**6 and 0.1 < Pr_avg < 2000:
            zeta    = 1+900/Re_avg
            lmbda   = (1.82*np.log(Re_avg)-1.64)**(-2)  
            h       = k_avg/self.Geometry.Tube.inner_diameter * ((0.125*zeta*Re_avg*Pr_avg*C_tem)/(lmbda+4.5*zeta**(0.5)*(Pr_avg**(0.666-1))))

        elif Re_avg > 10000 and 0.7 < Pr_avg < 160: # skal lige eftertjekkes
            h       = 0.023 * k_avg/self.Geometry.Tube.inner_diameter * Re_avg**(0.8)*Pr_avg**(0.4)

        else: 
            print("Error, out of correlation range")

        return h

    def twophase_heat_transfer(self,Node_in,Node_out):
    
        # Average fluid quality
        x_avg           = (Node_in.fluid.quality+Node_out.fluid.quality) / 2
        Node_in_liq     = Node_in.liquid_phase()
        Node_out_liq    = Node_out.liquid_phase()

        # Calculate average liquid velocity
        u_liq_avg   = self.m_dot / (( (Node_in_liq.density + Node_out_liq.density) / 2) * self.Geometry.Tube.inner_area)

        # Calculate average liquid properties
        rho_avg_liq = (Node_in_liq.density + Node_out_liq.density) / 2
        mu_avg_liq  = (Node_in_liq.dynamic_viscosity + Node_out_liq.dynamic_viscosity) / 2
        cp_avg_liq  = (Node_in_liq.specific_heat + Node_out_liq.specific_heat) / 2
        k_avg_liq   = (Node_in_liq.conductivity + Node_out_liq.conductivity) / 2

        # Average Re and Pr values for liquid only
        Re_avg_liq  = self.Reynolds_number(rho_avg_liq,u_liq_avg,mu_avg_liq,self.Geometry.Tube.inner_diameter)        

        if Re_avg_liq > 3000: 

            Pr_avg_liq  = self.Prandtl_number(cp_avg_liq,mu_avg_liq,k_avg_liq)

            # Heat transfer of liquid only 
            h_L         = 0.023*Re_avg_liq**0.8 *Pr_avg_liq**(0.4) *k_avg_liq/self.Geometry.Tube.inner_diameter

            # Critical pressure
            p_cr        = self.Node_in.fluid.critical_pressure
            
            # Average actual pressure
            p_avg       = (self.Node_in.fluid.pressure + self.Node_out.fluid.pressure) / 2

            # Reduced pressure
            p_r         = p_avg / p_cr
            
            # Two phase heat transfer coefficient
            h_TP        = h_L * ((1-x_avg)**(0.8)+(3.8*x_avg**(0.76)*(1-x_avg)**(0.04))/(p_r**(0.38)))        

        else:
            print("Error, laminar flow")

        return h_TP

    # ----- General functions for internal flow ----- #

    def Reynolds_number(self,rho,u,mu,d):
        Re = (rho * u * d) / mu
        return Re
    
    def Prandtl_number(self,cp,mu,k):
        Pr = (cp*mu)/k
        return Pr

class External: # External correlations
    def __init__(self,mass_flow,rownumber,node_in,node_out,geometry):
        self.z_2        = rownumber # Row number in the bank
        self.node_in    = node_in # Node object representing the fluid state
        self.node_out   = node_out # Node object representing the fluid state
        self.geometry   = geometry # Geometry class
        self.mass_flow  = mass_flow
        
    def heat_transfer(self):
        F = self.geometry.Free_flow_area
        
        d = self.geometry.outer_diameter
        
        avg_density = (self.node_in.fluid.density + self.node_out.fluid.density) / 2
        avg_Pr = (self.node_in.Prandtl_number + self.node_out.Prandtl_number) / 2
        u_g = self.mass_flow/(F*avg_density)
        
        avg_nu = (self.node_in.kinematic_viscosity + self.node_out.kinematic_viscosity) / 2
        avg_thermal_conductivity = (self.node_in.conductivity + self.node_out.conductivity) / 2
        
        avg_Re = (u_g * d) / avg_nu
                
        match self.geometry.Bank.arrangement:
            case arrangementType.Inline:
                X = 4 * (2 + self.geometry.Psi_r/7 - self.geometry.sigma_2) 
            case arrangementType.Staggered:
                X = self.geometry.sigma_1 / self.geometry.sigma_2  - 1.26 / self.geometry.Psi_r - 2
            case _: 
                NotImplementedError("Only Inline and Staggered tube arrangements are implemented for external heat transfer coefficient calculation.")
        
        n = 0.7 + 0.08 * np.tanh(X) + 0.005 * self.geometry.Psi_r
        C_q = (1.36-np.tanh(X))*( (1.1)/(self.geometry.Psi_r+8) - 0.014)
        
        # Calculate C_z based on number of rows and arrangement
        if self.geometry.Bank.transverse_number_of_rows >= 8:
            C_z = 1
        elif self.geometry.Bank.arrangement == arrangementType.Inline or (self.geometry.sigma_1/self.geometry.sigma_2) >= 2:
            C_z = 3.5 * self.geometry.Bank.transverse_number_of_rows**(0.03) - 2.72
        elif self.geometry.Bank.arrangement == arrangementType.Staggered and (self.geometry.sigma_1/self.geometry.sigma_2) < 2:
            C_z = 3.15 * self.geometry.Bank.transverse_number_of_rows**(0.05) - 2.5
        else:
            raise ValueError("Error in calculating C_z for external heat transfer coefficient.")
        
        # Determine external heat transfer coefficient:
        h_c = 1.13*C_z*C_q*avg_thermal_conductivity/d * avg_Re**n * avg_Pr**(1/3)
        
        # Determine the reduced heat transfer coefficient:
        
        beta = np.sqrt(2*h_c /( delta_r * k_r)  )
        psi_e = 1 - 0.016 * (self.geometry.finning_diameter/d-1)*(1+np.tanh(2*beta*l_r-1))

    def pressure_drop_1(self):
        
        # Free flow area
        F       = self.geometry.Free_flow_area
        d_eq    = self.geometry.Equivalent_diameter
        u_g     = self.mass_flow/(F*self.node.fluid.density)
        
        Re_eq   = (u_g*d_eq)/self.node.fluid.kinematic_viscosity
        print(Re_eq)
        
        if 5*10**3< Re_eq < 6 * 10**4:
            #Correction factor for small row numbers:
            if self.geometry.Bank.arrangement == 'staggered' and self.z_2 < 6:
                C_zm = np.exp(0.1*(6/self.z_2-1))
                print("succes")

            elif self.geometry.Bank.arrangement == 'inline' and self.z_2 < 6:
                C_zm = 1+(0.65/(self.z_2**3))
                print("succes")

            else:
                C_zm = 1

            # coefficient in similarity equation for aerodynamic resistance
            if self.geometry.Bank.arrangement == 'staggered':
                n = 0.17*(self.geometry.A_total_over_F)**(0.25) * (self.geometry.S_1_over_S_2)**(0.57) * np.exp(-0.36*(self.geometry.S_1_over_S_2))
                C_r = 2.8 * (self.geometry.A_total_over_F)**(0.53) * (self.geometry.S_1_over_S_2)**(1.30) * np.exp(-0.90*(self.geometry.S_1_over_S_2))
            elif self.geometry.Bank.arrangement == 'inline':
                if self.geometry.S_1_over_S_2 <= 2.1:
                    n = (self.geometry.A_total_over_F)**(0.08) * (0.184-0.088*(self.geometry.S_1_over_S_2))
                    C_r = 2.5 * (self.geometry.A_total_over_F)**(0.25) * np.exp(-1.70*(self.geometry.S_1_over_S_2))
                elif self.geometry.S_1_over_S_2 > 2.1:
                    n = 0
                    C_r = (self.geometry.A_total_over_F)**(0.10) * (0.132-0.016*(self.geometry.S_1_over_S_2))
            
            # Resistance coefficient is calculated:
            zeta_0 = C_zm * C_r * Re_eq**(-n)
    
            # Constant constant
            C_op = 1.1

            # Delta P calculation:
            delta_P = C_op * zeta_0 * self.z_2 * (self.node.fluid.density*u_g**2)/2 

        else:
            print("Error, flow out of correlation range")

        return(delta_P)
        # Phi parameter:

    def pressure_drop_2(self):
        pass

if __name__ == "__main__":
    # Geom definition
    Geom = Geometry()
    Geom.Tube.set_geometry(0.028,0)
    Geom.Duct.set_geometry(0.56,0.5)
    Geom.Fin.set_geometry(8e-4,0.0135,0.003)
    Geom.Bank.set_geometry(9,30,0.059,0.051,0.5,'staggered')
    
    P_in_air = (-4e3) + 966e2 
    T_in_air = 230
    
    P_in_water = 135e5 # bar to Pa
    T_in_water = 360
    
    
    # phase change check
    Node_1 = Node(FluidsList.Water,1,pf.Input.temperature(340),pf.Input.pressure(13500000))
    Node_2 = Node(FluidsList.Water,1,pf.Input.quality(0.9),pf.Input.pressure(13500000))
    
    Node_1 = Node(FluidsList.Water,1,pf.Input.temperature(340),pf.Input.pressure(13500000))
    Node_2 = Node(FluidsList.Water,1,pf.Input.quality(1),pf.Input.pressure(13500000))
    
    Node_internal_S1 = Node(FluidsList.Water,0,pf.Input.temperature(T_in_water),pf.Input.pressure(P_in_water))
    Node_internal_S2 = Node(FluidsList.Water,0,pf.Input.temperature(T_in_water-10),pf.Input.pressure(P_in_water))
    
    Node_external_S1 = Node(FluidsList.Air,1,pf.Input.temperature(T_in_air),pf.Input.pressure(P_in_air))
    Node_external_S2 = Node(FluidsList.Air,1,pf.Input.temperature(T_in_air+10),pf.Input.pressure(P_in_air))
    
    Internal_1 = Internal(0.5,Node_internal_S1,Node_internal_S2,Geom,0.5)
    print(Internal_1.heat_transfer())
    Internal_2 = Internal(0.5,Node_1,Node_2,Geom,0.5)
    print(Internal_2.heat_transfer())
    External = External(0.5,1,Node_external_S1,Node_external_S2,Geom)

16.666666666666668